# 🎭 Face Swap for Colab (photos & videos)

<a href="https://colab.research.google.com/github/nitrodamsel/faceswap-colab/blob/main/faceswap_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

One-image face swap with a **drag-and-drop UI** — upload a source face and a target photo/video, click **Swap**, download the result. No manual file paths.

**How to use**
1. `Runtime ▸ Change runtime type ▸ T4 GPU`
2. Run **cell 1** (setup — once).
3. Run **cell 2** — a UI appears inline (plus a public share link). Upload → Swap → Download.

**Engine:** insightface `inswapper_128` + optional `GFPGAN` enhancement, via `onnxruntime-gpu`.

**Why not the old roop_colab?** It pins Python-3.10-era `numpy`/`torch` and won't install on Colab's current Python 3.13. This notebook uses the pure-python **insightface 2.1** wheel and bakes in the **Tesla-T4 + cuDNN-9** fix (`cudnn_conv_algo_search=DEFAULT`) that otherwise silently drops inference to slow CPU.

> ⚠️ **Use responsibly.** Only swap faces that are your own or that you have permission to use. Don't make misleading or non-consensual content.

In [ ]:
#@title 1. Setup — install the face-swap engine (run once, ~2-3 min)
# roop / roop_colab is dead on modern Colab (Python 3.13). This uses insightface 2.1
# (a pure-python wheel) + onnxruntime-gpu, which install cleanly on 3.13.
!pip install -q insightface gradio
!pip uninstall -y onnxruntime -q
!pip install -q onnxruntime-gpu
print('\n✅ Setup done. Run the NEXT cell to open the Face Swap UI.')
print('   If it asks to restart (only when you ran other cells first):')
print('   Runtime > Restart session, then run ONLY the next cell.')


In [ ]:
#@title 🎭 Face Swap — run this, then use the UI below (upload & swap)
# One-image face swap for photos & videos.
# Engine: insightface inswapper_128 (+ optional GFPGAN enhance), GPU-accelerated.
import os, cv2, numpy as np, subprocess, urllib.request
import insightface
from insightface.app import FaceAnalysis
from insightface.utils import face_align
import onnxruntime as ort
import gradio as gr

MODELS = '/content/models'; os.makedirs(MODELS, exist_ok=True)
INSWAP = f'{MODELS}/inswapper_128.onnx'
GFP = f'{MODELS}/GFPGANv1.4.onnx'


def _dl(urls, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 1e6:
        return
    for u in urls:
        try:
            print('downloading', os.path.basename(dst), '...')
            urllib.request.urlretrieve(u, dst)
            if os.path.getsize(dst) > 1e6:
                return
        except Exception as e:
            print('  retry:', str(e)[:80])
    raise SystemExit('download failed: ' + dst)


_dl(['https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/inswapper_128.onnx',
     'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx'], INSWAP)

# --- providers: includes the Tesla-T4 + cuDNN-9 HEURISTIC_QUERY_FAILED workaround ---
_cuda = 'CUDAExecutionProvider' in ort.get_available_providers()
if _cuda:
    PROVIDERS = [('CUDAExecutionProvider', {'cudnn_conv_algo_search': 'DEFAULT', 'cudnn_conv_use_max_workspace': '1'}), 'CPUExecutionProvider']
    CTX = 0
else:
    PROVIDERS = ['CPUExecutionProvider']
    CTX = -1
print('Running on', 'GPU' if _cuda else 'CPU')

app = FaceAnalysis(name='buffalo_l', providers=PROVIDERS)
app.prepare(ctx_id=CTX, det_size=(640, 640))
swapper = insightface.model_zoo.get_model(INSWAP, providers=PROVIDERS)

_gfp = None
def _get_gfp():
    global _gfp
    if _gfp is None:
        _dl(['https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/facerestore_models/GFPGANv1.4.onnx'], GFP)
        _gfp = ort.InferenceSession(GFP, providers=PROVIDERS)
    return _gfp

_mask = np.zeros((512, 512), np.float32)
cv2.ellipse(_mask, (256, 256), (210, 260), 0, 0, 360, 1.0, -1)
_mask = cv2.GaussianBlur(_mask, (0, 0), 18)[:, :, None]


def _enhance(frame, kps):
    g = _get_gfp(); gin = g.get_inputs()[0].name
    M = face_align.estimate_norm(kps, 512)
    a = cv2.warpAffine(frame, M, (512, 512), borderMode=cv2.BORDER_REPLICATE)
    inp = ((a[:, :, ::-1].astype(np.float32) / 255.0 - 0.5) / 0.5).transpose(2, 0, 1)[None]
    o = g.run(None, {gin: inp})[0][0]
    o = np.clip(o, -1, 1); o = ((o + 1) / 2).transpose(1, 2, 0)[:, :, ::-1] * 255.0
    IM = cv2.invertAffineTransform(M)
    w = cv2.warpAffine(o.astype(np.float32), IM, (frame.shape[1], frame.shape[0]))
    wm = cv2.warpAffine(_mask, IM, (frame.shape[1], frame.shape[0]))
    wm = wm[:, :, None] if wm.ndim == 2 else wm
    return (w * wm + frame.astype(np.float32) * (1 - wm)).astype(np.uint8)


def _process(frame, src_face, enhance, many):
    faces = app.get(frame)
    if not faces:
        return frame
    tgts = faces if many else [sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))[-1]]
    for tf in tgts:
        frame = swapper.get(frame, tf, src_face, paste_back=True)
    if enhance:
        for tf in tgts:
            frame = _enhance(frame, tf.kps)
    return frame


def swap(source_path, target_path, enhance, many, progress=gr.Progress()):
    if not source_path or not target_path:
        raise gr.Error('Please provide both a source face and a target file.')
    sf = app.get(cv2.imread(source_path))
    if not sf:
        raise gr.Error('No face detected in the source image.')
    src_face = sorted(sf, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))[-1]
    ext = os.path.splitext(target_path)[1].lower()
    if ext in ('.jpg', '.jpeg', '.png', '.bmp', '.webp'):
        out = '/content/out.jpg'
        cv2.imwrite(out, _process(cv2.imread(target_path), src_face, enhance, many))
        return None, out, out
    cap = cv2.VideoCapture(target_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(3)); h = int(cap.get(4)); total = int(cap.get(7)) or 0
    tmp = '/content/_noaudio.mp4'
    writer = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    i = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        writer.write(_process(fr, src_face, enhance, many)); i += 1
        if total:
            progress(i / total, desc=f'frame {i}/{total}')
    cap.release(); writer.release()
    out = '/content/out.mp4'
    subprocess.run(['ffmpeg', '-y', '-i', tmp, '-i', target_path, '-map', '0:v:0', '-map', '1:a:0?',
                    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-preset', 'veryfast',
                    '-c:a', 'aac', '-shortest', out], capture_output=True)
    return out, None, out


with gr.Blocks(title='Face Swap', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎭 Face Swap\nUpload a **source face** and a **target photo or video**, then click **Swap**. '
                'For consensual, fun use only — swap faces you own or have permission to use.')
    with gr.Row():
        with gr.Column():
            src = gr.Image(type='filepath', label='Source face (the face to put ON the target)', height=280)
            tgt = gr.File(label='Target photo or video', file_types=['image', 'video'])
            with gr.Row():
                enh = gr.Checkbox(label='Enhance (GFPGAN — sharper, slower)', value=False)
                many = gr.Checkbox(label='Swap all faces in frame', value=True)
            go = gr.Button('🔄 Swap', variant='primary')
        with gr.Column():
            ov = gr.Video(label='Result (video)')
            oi = gr.Image(label='Result (photo)')
            od = gr.File(label='⬇️ Download result')
    go.click(swap, inputs=[src, tgt, enh, many], outputs=[ov, oi, od])

demo.launch(share=True, debug=False)
